# GPT-1 구조 기반 챗봇 언어모델 사전학습 프로젝트

## 프로젝트 소개

이 프로젝트는 기존 encoder-decoder Transformer 챗봇 코드를 GPT-1 방식의 decoder-only 언어 모델로 바꾸는 실습입니다. ChatbotData의 질문과 답변을 하나의 토큰 시퀀스로 만들고, 이전 토큰만 보고 다음 토큰을 예측하도록 학습합니다.

- 데이터: 한국어 ChatbotData Q/A 11,823쌍
- 학습 목표: 다음 토큰 예측 언어 모델 손실 L1
- 최종 모델: 4-layer GPT, `D_MODEL=256`, 학습 가능한 위치 임베딩, weight tying
- 생성 방식: `temperature=0.7`, `top_k=10` sampling

## Original Transformer → GPT-1 아키텍처 변경사항

| 기존 seq2seq Transformer | 본 프로젝트의 GPT 구조 |
| --- | --- |
| Encoder와 Decoder를 모두 사용 | Encoder를 제거하고 Decoder 블록만 사용 |
| Decoder가 encoder 출력에 cross-attention | Cross-attention을 제거하고 masked self-attention만 사용 |
| 질문과 답변을 별도 입력으로 사용 | `<bos> Q <sep> A <eos>`를 하나의 입력으로 연결 |
| sinusoidal position encoding | 학습 가능한 position embedding |
| ReLU 활성화 함수 | GELU 활성화 함수 |
| Greedy decoding | 특수 토큰 차단 + temperature/top-k sampling |

### 1. Encoder 삭제

- 기존 Transformer의 Encoder 전체를 제거합니다.
- `EncoderLayer`, `Encoder` 클래스가 불필요합니다.
- Decoder 블록만 반복하는 decoder-only 구조로 구성합니다.

### 2. Cross-Attention 삭제

- Decoder가 encoder 출력을 참조하는 cross-attention을 제거합니다.
- 각 GPT block은 `Masked Self-Attention → FFN`의 두 서브레이어로 구성됩니다.
- 이에 따라 LayerNorm도 세 개에서 두 개로 줄어듭니다.

### 3. 위치 임베딩 변경

- 고정된 sinusoidal position encoding 대신 학습 가능한 `nn.Embedding`을 사용합니다.
- 토큰 임베딩과 위치 임베딩을 더해 첫 번째 hidden state를 구성합니다.

### 4. 활성화 함수 변경

- position-wise FFN의 `ReLU`를 GPT-1에서 사용한 `GELU`로 변경합니다.

### 5. 입력 및 학습 목표 변경

- seq2seq의 encoder input / decoder input 분리 구조를 단일 시퀀스 입력 구조로 변경합니다.
- 챗봇 Q/A는 `<bos> Q <sep> A <eos>` 형태로 직렬화합니다.
- `input = seq[:-1]`, `target = seq[1:]`로 한 칸 shift하여 다음 토큰을 예측합니다.
- `<sep>`는 질문과 답변의 경계를 나타내며, 학습 시 다음 토큰 예측 대상에도 포함됩니다.

### 6. 출력층 변경: Weight Tying

- 토큰 임베딩 행렬을 출력 언어 모델 head와 공유합니다.
- `self.fc.weight = self.token_emb.weight`로 구현합니다.

### 7. 마스크 변경

- encoder mask와 decoder-encoder mask를 제거합니다.
- 미래 토큰을 보지 못하게 하는 causal mask와 PAD mask만 사용합니다.

## 코드 흐름

```mermaid
flowchart TD
    A["ChatbotData Q/A"] --> B["텍스트 정제"]
    B --> C["SentencePiece BPE 학습"]
    C --> D["BOS + Q + SEP + A + EOS 시퀀스 구성"]
    D --> E["input = seq[:-1], target = seq[1:]"]
    E --> F["Token Embedding + Position Embedding"]
    F --> G["GPTBlock x 4: Causal Self-Attention + FFN"]
    G --> H["Weight-tied LM Head"]
    H --> I["Cross-Entropy 다음 토큰 손실"]
    I --> J["Adam + Noam LR Scheduler 학습"]
    J --> K["SEP 조건부 Top-k 답변 생성"]
```

### 라이브러리 설치 및 버전 확인

In [1]:
# BPE 토크나이저 학습에 필요한 SentencePiece를 현재 커널에 설치합니다.
import sys
!"{sys.executable}" -m pip install -q sentencepiece

In [2]:
# 데이터 처리, 모델 구현, 파일 다운로드에 사용할 라이브러리를 불러옵니다.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import re
import urllib.request

print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'torch: {torch.__version__}')

numpy: 2.5.1
pandas: 3.0.3
torch: 2.11.0+cu128


---
## Step 1. 데이터 다운로드

In [3]:
# 데이터와 토크나이저 파일을 저장할 로컬 디렉터리를 준비합니다.
os.makedirs('./data', exist_ok=True)

# 파일이 없을 때만 공개 ChatbotData를 내려받아 재실행 비용을 줄입니다.
csv_path = './data/ChatbotData.csv'
if not os.path.exists(csv_path):
    url = 'https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv'
    print('다운로드 중...')
    urllib.request.urlretrieve(url, csv_path)
    print('완료!')

# 질문(Q)과 답변(A) 열만 추출해 이후의 단일 GPT 시퀀스 구성에 사용합니다.
df = pd.read_csv(csv_path)
questions = list(df['Q'])
answers   = list(df['A'])

print(f'총 쌍 수: {len(questions)}')
df.head()

총 쌍 수: 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


---
## Step 2. 데이터 전처리

GPT는 Decoder-Only 자기회귀 모델이므로, Q-A 쌍을 **하나의 연속 시퀀스**로 이어붙입니다.
- 이번 과제는 pretrain만 고려하므로, 분류 헤드 없이 다음 토큰 예측(L₁)만 수행합니다.
- `<bos> Q <sep> A <eos>` 형태로 구성합니다. `<sep>` 토큰으로 Q/A 경계를 명확히 합니다.
- 원 논문의 raw-text 사전학습은 구분자 없는 연속 텍스트를 사용합니다. 본 과제에서는 챗봇 Q/A를 생성 모델 입력으로 직렬화하기 위해 `<sep>`를 추가하며, 이 토큰도 다음 토큰 예측 손실로 학습됩니다.

In [4]:
# 토크나이저 학습과 모델 입력에 동일하게 적용하는 정규화 함수입니다.
def preprocess_sentence(sentence):
    # 영문 대소문자를 통일하고, 학습하지 않을 문자는 공백으로 바꿉니다.
    sentence = str(sentence).lower()
    sentence = re.sub(r'[^가-힣a-z0-9?.!,\s]', ' ', sentence)
    sentence = re.sub(r'([?.!,])', r' \1 ', sentence)
    sentence = re.sub(r'\s+', ' ', sentence)
    return sentence.strip()

for q, a in zip(questions[:3], answers[:3]):
    print(f'Q: {preprocess_sentence(q)}')
    print(f'A: {preprocess_sentence(a)}')
    print()

Q: 12시 땡 !
A: 하루가 또 가네요 .

Q: 1지망 학교 떨어졌어
A: 위로해 드립니다 .

Q: 3박4일 놀러가고 싶다
A: 여행은 언제나 좋죠 .



---
## Step 3. 토큰화 (SentencePiece BPE)

- `--model_type=bpe` 옵션으로 GPT 논문과 동일한 BPE 토큰화를 사용합니다.
- `--user_defined_symbols=<sep>` 옵션으로 Q/A 구분자 토큰을 추가합니다.
- 코퍼스를 Q+A 연결 형태로 작성하여 실제 학습 데이터와 일관성을 유지합니다.

In [5]:
import sentencepiece as spm

VOCAB_SIZE   = 8000
corpus_path  = './data/chatbot_corpus.txt'
model_prefix = './data/chatbot_spm'

# [수정] 코퍼스를 Q+A 연결 형태로 작성 (실제 학습 데이터와 일관성 유지)
with open(corpus_path, 'w', encoding='utf-8') as f:
    for q, a in zip(questions, answers):
        f.write(preprocess_sentence(q) + ' <sep> ' + preprocess_sentence(a) + '\n')

spm.SentencePieceTrainer.Train(
    f'--input={corpus_path} '
    f'--model_prefix={model_prefix} '
    f'--vocab_size={VOCAB_SIZE} '
    f'--model_type=bpe '                   # [수정] 기본 Unigram → BPE (논문 일치)
    f'--user_defined_symbols=<sep> '        # [추가] Q/A 구분자 토큰
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    f'--character_coverage=0.9995'
)

tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(model_prefix + '.model')

PAD_ID = tokenizer.pad_id()   # 0
BOS_ID = tokenizer.bos_id()   # 1
EOS_ID = tokenizer.eos_id()   # 2
SEP_ID = tokenizer.piece_to_id('<sep>')  # Q/A 구분자
assert SEP_ID != tokenizer.unk_id(), '<sep> 토큰 등록에 실패했습니다.'

# [수정] 실제 vocab 크기를 tokenizer에서 가져옴 (요청값과 다를 수 있음)
ACTUAL_VOCAB_SIZE = tokenizer.get_piece_size()

print(f'vocab size: {ACTUAL_VOCAB_SIZE}')
print(f'PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}, SEP={SEP_ID}')
print('예시:', tokenizer.encode_as_pieces('오늘 날씨가 어때요?'))
print('ids :', tokenizer.encode_as_ids('오늘 날씨가 어때요?'))

vocab size: 8000
PAD=0, BOS=1, EOS=2, SEP=4
예시: ['▁오늘', '▁날씨가', '▁어때요', '?']
ids : [130, 2013, 2130, 6970]


---
## Step 4. GPT용 데이터셋 구성

**변경점**: seq2seq에서는 src(encoder 입력)와 tgt(decoder 입력)를 분리했지만,
GPT는 **하나의 시퀀스**로 구성합니다.

```
seq2seq:  src = [Q 토큰들]          tgt = [<bos> A 토큰들 <eos>]
GPT:      seq = [<bos> Q 토큰들 <sep> A 토큰들 <eos>]
          input  = seq[:-1]         (마지막 토큰 제외)
          target = seq[1:]          (첫 토큰 제외 — 한 칸 shift)
```

In [6]:
# 한 Q/A 예제의 최대 길이입니다. 길이가 긴 예제는 제외하고 짧은 예제는 PAD로 채웁니다.
MAX_LEN = 80

def build_gpt_sequences(questions, answers, tokenizer, max_len):
    """Q-A 쌍을 <bos> Q <sep> A <eos> 형태의 단일 시퀀스로 변환"""
    sequences = []
    for q, a in zip(questions, answers):
        q_ids = tokenizer.encode_as_ids(preprocess_sentence(q))
        a_ids = tokenizer.encode_as_ids(preprocess_sentence(a))
        # [수정] <bos> + Q + <sep> + A + <eos>
        seq = [BOS_ID] + q_ids + [SEP_ID] + a_ids + [EOS_ID]
        if len(seq) <= max_len:
            seq = seq + [PAD_ID] * (max_len - len(seq))
            sequences.append(seq)
    return sequences

sequences = build_gpt_sequences(questions, answers, tokenizer, MAX_LEN)
print(f'유효 시퀀스 수: {len(sequences)}')
print(f'시퀀스 길이: {len(sequences[0])}')

# 예시 확인
print('\n--- 예시 시퀀스 ---')
example = sequences[0]
print('토큰 IDs:', example[:20], '...')
non_special = [t for t in example if t != PAD_ID]
print('디코딩:  ', tokenizer.decode_ids(non_special))

# SEP 위치 확인
sep_pos = example.index(SEP_ID)
print(f'SEP 위치: {sep_pos}')
print(f'Q 부분: {tokenizer.decode_ids(example[1:sep_pos])}')
print(f'A 부분: {tokenizer.decode_ids(example[sep_pos+1:example.index(EOS_ID)])}')

유효 시퀀스 수: 11823
시퀀스 길이: 80

--- 예시 시퀀스 ---
토큰 IDs: [1, 5551, 6982, 3200, 109, 4, 4485, 215, 5919, 5, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0] ...
디코딩:   12시 땡 !<sep> 하루가 또 가네요 .
SEP 위치: 5
Q 부분: 12시 땡 !
A 부분: 하루가 또 가네요 .


In [7]:
# 고정 길이 시퀀스를 텐서로 변환하고, mini-batch 단위로 섞어 공급합니다.
from torch.utils.data import TensorDataset, DataLoader

seq_tensor = torch.tensor(sequences, dtype=torch.long)

# input: seq[:-1], target: seq[1:] (한 토큰 shift)
input_tensor  = seq_tensor[:, :-1]
target_tensor = seq_tensor[:, 1:]

# 입력 구성 검증: target은 input보다 정확히 한 토큰 오른쪽으로 이동해야 합니다.
assert torch.equal(input_tensor[:, 1:], target_tensor[:, :-1])
# 모든 학습 예제에 Q/A 경계 토큰이 포함되는지 확인합니다.
assert (input_tensor == SEP_ID).any(dim=1).all()

print(f'input shape:  {input_tensor.shape}')
print(f'target shape: {target_tensor.shape}')

# shuffle=True로 매 epoch마다 Q/A 예제 순서를 섞되, 한 시퀀스 내부의 토큰 순서는 유지합니다.
BATCH_SIZE = 64
train_dataset    = TensorDataset(input_tensor, target_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
print(f'배치 수: {len(train_dataloader)}')

input shape:  torch.Size([11823, 79])
target shape: torch.Size([11823, 79])
배치 수: 185


---
## Step 5. GPT 모델 구성

기존 Transformer 코드를 수정하여 GPT-1 모델을 구성합니다.

In [8]:
# GPU가 있으면 CUDA에서 학습하고, 없으면 CPU에서 동일한 코드를 실행합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [9]:
# === MultiHeadAttention: 변경 없음 ===

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        # 각 head가 담당할 feature 차원입니다. d_model은 num_heads로 나누어떨어져야 합니다.
        self.depth     = d_model // num_heads
        self.d_model   = d_model
        self.W_q    = nn.Linear(d_model, d_model)
        self.W_k    = nn.Linear(d_model, d_model)
        self.W_v    = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # QK^T / sqrt(d_k)로 토큰 간 attention score를 계산합니다.
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(Q.size(-1))
        if mask is not None:
            # causal/PAD 위치의 score를 매우 작은 값으로 만들어 softmax 확률을 0에 가깝게 합니다.
            scores = scores + (mask * -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V), attn

    def split_heads(self, x):
        b, s, _ = x.size()
        # [B, S, D]를 [B, heads, S, depth]로 바꿔 head별 attention을 병렬 계산합니다.
        return x.view(b, s, self.num_heads, self.depth).permute(0, 2, 1, 3)

    def combine_heads(self, x):
        b, _, s, _ = x.size()
        return x.permute(0, 2, 1, 3).contiguous().view(b, s, self.d_model)

    def forward(self, Q, K, V, mask=None):
        out, attn = self.scaled_dot_product_attention(
            self.split_heads(self.W_q(Q)),
            self.split_heads(self.W_k(K)),
            self.split_heads(self.W_v(V)), mask)
        return self.linear(self.combine_heads(out)), attn

print('MultiHeadAttention 정의 완료')

MultiHeadAttention 정의 완료


In [10]:
# === PoswiseFeedForwardNet ===
# [변경] ReLU → GELU

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.fc2  = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()  # [변경] ReLU → GELU

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))  # [변경] relu → gelu

print('PoswiseFeedForwardNet 정의 완료 (GELU 적용)')

PoswiseFeedForwardNet 정의 완료 (GELU 적용)


In [11]:
# === GPTBlock (기존 DecoderLayer 수정) ===
# [삭제] Cross-Attention (enc_dec_attn) 제거
# [삭제] norm_3 제거 (서브레이어 3개 → 2개)
# [변경] forward 시그니처에서 enc_out, dec_enc_mask 제거

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        # [삭제] self.enc_dec_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn    = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        # [삭제] self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)
        self.do     = nn.Dropout(dropout)

    def forward(self, x, mask):
        # [변경] enc_out, dec_enc_mask 파라미터 제거
        # Pre-LN 구조: 정규화한 입력으로 masked self-attention을 계산한 뒤 residual을 더합니다.
        n = self.norm_1(x)
        out, attn = self.self_attn(n, n, n, mask=mask)
        out = self.do(out) + x
        residual = out
        # [삭제] Cross-Attention 블록 전체 제거
        # 두 번째 Pre-LN 서브레이어는 position-wise FFN이며, 다시 residual connection을 사용합니다.
        out = self.do(self.ffn(self.norm_2(out))) + residual
        return out, attn

print('GPTBlock 정의 완료 (Cross-Attention 제거)')

GPTBlock 정의 완료 (Cross-Attention 제거)


In [12]:
# === GPT 모델 (기존 Transformer 수정) ===
# [삭제] Encoder 전체 제거
# [삭제] positional_encoding() 함수 제거
# [변경] sinusoidal 위치 임베딩 → nn.Embedding (학습 가능)
# [변경] forward에서 enc_in, enc_mask, dec_enc_mask 제거 → 단일 입력

class GPT(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 vocab_size, max_seq_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # PAD embedding은 갱신하지 않고, 일반 토큰 embedding만 학습합니다.
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # [변경] sinusoidal → 학습 가능한 위치 임베딩
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)

        self.do = nn.Dropout(dropout)

        # [삭제] self.encoder = Encoder(...)
        # [변경] Decoder → GPTBlock 스택
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

        self.fc = nn.Linear(d_model, vocab_size, bias=False)
        # 입력 embedding 행렬을 출력 LM head에도 공유해 파라미터를 줄이고 GPT-1 식을 따릅니다.
        self.fc.weight = self.token_emb.weight  # Weight Tying

    def forward(self, x, mask):
        # [변경] enc_in, dec_in 2개 입력 → x 1개 입력
        b, seq_len = x.size()

        # [변경] 위치 인덱스 생성 + 학습 가능한 위치 임베딩
        # 배치의 모든 예제에 0..seq_len-1 위치 ID를 만들어 토큰 순서 정보를 더합니다.
        positions = torch.arange(seq_len, device=x.device)
        h = self.do(self.token_emb(x) + self.pos_emb(positions))  # h₀ = UWₑ + Wₚ

        # [삭제] encoder forward 전체
        # [변경] decoder forward → GPTBlock 스택 forward
        attns = []
        for block in self.blocks:
            h, attn = block(h, mask)
            attns.append(attn)

        h = self.final_norm(h)
        logits = self.fc(h)  # P(u) = softmax(hₙ Wₑᵀ)
        return logits, attns

print('GPT 모델 정의 완료')

GPT 모델 정의 완료


In [13]:
# === 마스크 생성 ===
# [삭제] enc_mask, dec_enc_mask 생성 제거
# [변경] causal mask + padding mask만 사용

def generate_gpt_mask(seq):
    """GPT용 마스크: causal (look-ahead) + padding"""
    seq_len = seq.size(1)
    # causal mask: 미래 토큰을 볼 수 없도록
    # 상삼각(미래 위치)을 1로 만들어 현재 토큰이 미래 토큰을 보지 못하게 합니다.
    causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).to(seq.device)
    # padding mask
    # PAD를 key로 참조하지 않도록 [B, 1, 1, S] 모양의 mask를 만듭니다.
    pad_mask = (seq == PAD_ID).unsqueeze(1).unsqueeze(1).float()
    # 둘 중 하나라도 마스킹되면 마스킹
    mask = torch.max(pad_mask, causal_mask.unsqueeze(0).unsqueeze(0))
    return mask

print('마스크 생성 함수 정의 완료')

마스크 생성 함수 정의 완료


---
## Step 6. 모델 생성 및 확인

In [14]:
# 축소 GPT-1 실험의 최종 하이퍼파라미터입니다.
D_MODEL  = 256
N_LAYERS = 4
N_HEADS  = 8
D_FF     = 512
DROPOUT  = 0.1

# 실제 tokenizer vocabulary 크기와 최대 입력 길이를 모델에 전달합니다.
model = GPT(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    vocab_size=ACTUAL_VOCAB_SIZE,  # [수정] tokenizer.get_piece_size() 사용
    max_seq_len=MAX_LEN,
    dropout=DROPOUT
).to(device)

print(model)
print(f'\n파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

GPT(
  (token_emb): Embedding(8000, 256, padding_idx=0)
  (pos_emb): Embedding(80, 256)
  (do): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x GPTBlock(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (linear): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): PoswiseFeedForwardNet(
        (fc1): Linear(in_features=256, out_features=512, bias=True)
        (fc2): Linear(in_features=512, out_features=256, bias=True)
        (gelu): GELU(approximate='none')
      )
      (norm_1): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
      (norm_2): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
      (do): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
  (fc): Linear(in_fe

In [15]:
# DataLoader에서 한 배치를 꺼내 GPT의 입력/출력 차원과 공유 가중치를 확인합니다.
sample_batch = next(iter(train_dataloader))
sample_input, sample_target = sample_batch

print(f'input shape:  {sample_input.shape}')   # [batch, seq_len-1]
print(f'target shape: {sample_target.shape}')   # [batch, seq_len-1]

# 모델 forward 테스트
sample_mask = generate_gpt_mask(sample_input.to(device))
sample_output, _ = model(sample_input.to(device), sample_mask)
print(f'output shape: {sample_output.shape}')  # [batch, seq_len-1, vocab_size]

# 모델 입력/출력 및 weight tying 검증
assert sample_input.size(1) <= model.pos_emb.num_embeddings
assert sample_output.shape == (sample_input.size(0), sample_input.size(1), ACTUAL_VOCAB_SIZE)
assert model.fc.weight.data_ptr() == model.token_emb.weight.data_ptr()
print('입력 shift, 출력 shape, 위치 임베딩 범위, weight tying 검증 통과')

input shape:  torch.Size([64, 79])
target shape: torch.Size([64, 79])
output shape: torch.Size([64, 79, 8000])
입력 shift, 출력 shape, 위치 임베딩 범위, weight tying 검증 통과


---
## Step 7. 학습

최종 실험 설정은 4 layers, D_MODEL=256, DROPOUT=0.1, 50 epochs입니다.  
NLP03의 Noam learning-rate schedule을 적용하여 배치마다 warmup 및 decay를 수행합니다. 이 설정에서는 총 9,250 update 중 4,000 update까지 warmup이 진행됩니다.

In [16]:
# 손실 함수: L₁ — 다음 토큰 예측 (CrossEntropyLoss, 패딩 무시)
def loss_function(target, pred):
    # [B, S, V] logits와 [B, S] target을 평탄화해 각 위치의 next-token loss를 구합니다.
    loss_ = F.cross_entropy(
        pred.contiguous().view(-1, pred.size(-1)),
        target.contiguous().view(-1),
        reduction='none'
    ).view(target.size())
    # PAD는 실제 언어 토큰이 아니므로 평균 loss 계산에서 제외합니다.
    mask = (target != PAD_ID).float()
    return (loss_ * mask).sum() / mask.sum()


def train_step(inp, target, model, optimizer):
    model.train()
    optimizer.zero_grad()

    # 입력마다 causal + padding mask를 생성해 미래 토큰 누설을 막습니다.
    mask = generate_gpt_mask(inp.to(device))
    preds, _ = model(inp.to(device), mask)
    loss = loss_function(target.to(device), preds)

    # 역전파로 gradient를 계산하고 Adam이 파라미터를 한 번 갱신합니다.
    loss.backward()
    optimizer.step()
    return loss

print('학습 함수 정의 완료')

학습 함수 정의 완료


In [17]:
# NLP03과 동일한 Noam learning-rate schedule: warmup 동안 증가 후 역제곱근 비율로 감소
class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        # _LRScheduler의 epoch index를 실제 1-based update step으로 변환합니다.
        step = max(1, self.last_epoch + 1)
        lr = (self.d_model ** -0.5) * min(
            step ** -0.5, step * (self.warmup_steps ** -1.5)
        )
        return [lr for _ in self.base_lrs]

# scheduler가 학습률을 결정하므로 초기 lr은 작은 값으로 둡니다.
# 초기 lr은 scheduler가 곧 덮어쓰며, Adam의 beta/epsilon은 Transformer 계열의 기본 설정을 사용합니다.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-9, betas=(0.9, 0.98), eps=1e-9)
lr_scheduler = LearningRateScheduler(optimizer, d_model=D_MODEL, warmup_steps=4000)

EPOCHS = 50

In [18]:
from tqdm.notebook import tqdm

# epoch는 전체 Q/A 데이터셋을 한 번 순회하는 단위입니다.
for epoch in range(EPOCHS):
    total_loss, n = 0.0, len(train_dataloader)
    pbar = tqdm(total=n, desc=f'Epoch {epoch+1:2d}/{EPOCHS}')

    for inp, target in train_dataloader:
        # 한 mini-batch에 대해 forward → loss → backward → optimizer update를 수행합니다.
        loss = train_step(inp, target, model, optimizer)
        # train_step 내부의 optimizer.step() 직후 배치 단위로 LR을 갱신합니다.
        lr_scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        pbar.update(1)

    pbar.close()
    print(f'Epoch {epoch+1:2d} -- avg loss: {total_loss/n:.4f}, lr: {optimizer.param_groups[0]["lr"]:.2e}')

Epoch  1/50:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch  1 -- avg loss: 106.8571, lr: 4.60e-05


Epoch  2/50:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch  2 -- avg loss: 31.2242, lr: 9.17e-05


Epoch  3/50:   0%|          | 0/185 [00:00<?, ?it/s]

...

Epoch 48 -- avg loss: 2.9975, lr: 6.63e-04


Epoch 49/50:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch 49 -- avg loss: 2.9263, lr: 6.56e-04


Epoch 50/50:   0%|          | 0/185 [00:00<?, ?it/s]

Epoch 50 -- avg loss: 2.8581, lr: 6.50e-04


---
## Step 8. 텍스트 생성 (출력 확인)

생성 시 &lt;pad&gt;, &lt;bos&gt;, &lt;sep&gt;는 후보에서 제외합니다. Greedy decoding 대신 temperature=0.7, top_k=10 sampling을 사용해 반복을 줄이고 안정적인 답변을 생성합니다.

In [25]:
def generate(prompt, model, tokenizer, max_gen=40, temperature=0.7, top_k=10):
    """질문 뒤의 <sep>를 조건으로 temperature=0.7, top-k=10 답변을 생성합니다."""
    if temperature <= 0:
        raise ValueError('temperature는 0보다 커야 합니다.')
    if not 1 <= top_k <= ACTUAL_VOCAB_SIZE:
        raise ValueError(f'top_k는 1~{ACTUAL_VOCAB_SIZE} 범위여야 합니다.')

    # Dropout을 끄고 동일한 모델에서 autoregressive generation을 수행합니다.
    model.eval()
    prompt = preprocess_sentence(prompt)
    # 학습 입력과 동일하게 질문 뒤에 Q/A 경계 토큰을 제공합니다.
    ids = [BOS_ID] + tokenizer.encode_as_ids(prompt) + [SEP_ID]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    answer_ids = []

    with torch.no_grad():
        for _ in range(max_gen):
            # [추가] 위치 임베딩 범위 초과 방지
            if input_ids.size(1) >= model.pos_emb.num_embeddings:
                break
            mask = generate_gpt_mask(input_ids)
            logits, _ = model(input_ids, mask)
            next_token_logits = logits[0, -1].clone()

            # 답변 본문에 나타나면 안 되는 구조용 특수 토큰은 생성 후보에서 제외합니다.
            next_token_logits[PAD_ID] = -float('inf')
            next_token_logits[BOS_ID] = -float('inf')
            next_token_logits[SEP_ID] = -float('inf')

            # greedy argmax 대신 temperature + top-k sampling을 적용합니다.
            next_token_logits = next_token_logits / temperature
            # 상위 k개 후보 안에서만 확률적으로 하나를 고릅니다.
            topk_logits, topk_ids = torch.topk(next_token_logits, top_k)
            probs = F.softmax(topk_logits, dim=-1)
            next_id = topk_ids[torch.multinomial(probs, 1)].item()
            # EOS가 선택되면 답변을 종료합니다.
            if next_id == EOS_ID:
                break
            answer_ids.append(next_id)
            input_ids = torch.cat(
                [input_ids, torch.tensor([[next_id]], device=device)], dim=1
            )

    return tokenizer.decode_ids(answer_ids)


test_prompts = [
    '안녕하세요',
    '오늘 기분이 어때요?',
    '배가 고파요',
    '심심해요',
    '사랑해',
]

print('=== GPT 텍스트 생성 테스트 ===')
for prompt in test_prompts:
    result = generate(prompt, model, tokenizer)
    print(f'입력: {prompt}')
    print(f'출력: {result}')
    print()

=== GPT 텍스트 생성 테스트 ===
입력: 안녕하세요
출력: 안녕하세요 .

입력: 오늘 기분이 어때요?
출력: 저도 해보고 싶은 알바예요 .

입력: 배가 고파요
출력: 산책 좀 해야겠네여 .

입력: 심심해요
출력: 저랑 놀아요 .

입력: 사랑해
출력: 많이 힘들었겠어요 .



---
## Step 9. 결과 분석 및 회고

### 결과 분석

- 최종 모델은 4 layers, D_MODEL=256, N_HEADS=8, D_FF=512로 구성되며 파라미터 수는 **4,177,408개**입니다.
- 50 epoch 학습에서 평균 loss는 **106.8571 → 2.8581**로 안정적으로 감소했습니다. Noam scheduler의 learning rate도 warmup 후 감소 구간으로 전환되어 학습이 발산하지 않았습니다.
- temperature=0.7, top_k=10 생성에서는 심심해요라는 입력에 저랑 놀아요.와 같이 입력 의도에 맞는 답변이 생성되었습니다. 일부 입력에서는 의미가 어색한 문장이 남아 있으며, 이는 작은 챗봇 데이터와 축소된 모델 용량의 한계입니다.

### 프로젝트에서 수정한 점

1. Encoder와 cross-attention을 제거해 decoder-only GPT 구조로 변경했습니다.
2. Q/A를 &lt;bos&gt; Q &lt;sep&gt; A &lt;eos&gt; 단일 시퀀스로 직렬화하고, 다음 토큰 예측을 위한 shift target을 구성했습니다.
3. SentencePiece를 BPE로 학습하고 &lt;sep&gt;를 사용자 정의 토큰으로 등록했습니다.
4. 학습 가능한 position embedding, GELU, causal mask, weight tying을 적용했습니다.
5. 입력 shift·경계 토큰·출력 shape·weight tying을 assertion으로 검증했습니다.
6. Noam learning-rate schedule, 특수 토큰 생성 차단, temperature/top-k sampling을 추가했습니다.

### 회고

- Decoder-only 모델에서는 입력을 하나의 연속 시퀀스로 만들고, 미래 토큰을 가리는 causal mask가 핵심이라는 점을 확인했습니다.
- &lt;sep&gt;는 질문과 답변의 경계를 모델에 알려 주지만, 생성 후보에서는 제외해야 답변 내부로 누출되지 않습니다.
- 2-layer/30-epoch 실험보다 4-layer/50-epoch 실험에서 loss와 생성 품질이 개선되었습니다. 다만 검증셋을 분리하지 않았으므로 loss만으로 일반화 성능을 판단할 수는 없습니다.
- 원 GPT-1은 BooksCorpus의 긴 연속 텍스트로 사전학습했습니다. 본 프로젝트는 과제 조건에 맞춰 챗봇 Q/A를 직렬화한 축소 실험이므로, 더 좋은 품질을 위해서는 더 큰 연속 텍스트 코퍼스와 검증셋이 필요합니다.


1만여개의 작은 데이터로도 어느정도 반응은 유도할 수 있도록 학습이 되었다. 데이터의 규모를 생각해서 layer를 4로 제한하고 과적합 방지를 위해 epoch도 50까지만 돌려서 학습을 돌려보았다. 그런데 데이터가 적을 땐 gpt의 성능이 떨어지는 것으로 아는데 기존 seq2seq와 어느정도 차이가 있는지 궁금해졌다.